# 🐱🐶 Cats vs Dogs — Binary Image Classification with CNNs

**Full pipeline:**

| Step | What happens |
|---|---|
| 1 | Data loading & generators (prefilled) |
| 2 | Dataset inspection — class balance & sample grid |
| 3 | CNN architecture definition & rationale |
| 4 | Optimization setup |
| 5 | Train with augmentation + learning curves |
| 6 | Evaluate — confusion matrix, precision, recall |
| 7 | Inference on unlabeled test set → CSV export |
| 8 | Baseline (no augmentation) comparison |
| 9 | Class imbalance handling with class weights |
| 10 | Save model + training config |
| 11 | Extension — MobileNetV2 transfer learning |
| 12 | Deliverables checklist |

> **Setup:** Download the [Kaggle Dogs vs Cats dataset](https://www.kaggle.com/c/dogs-vs-cats/data).  
> Extract and rename the folder to `cats_dogs`, place it inside a `data/` folder.  
> Upload `data/` to your Colab session or mount Google Drive.
>
> **Runtime:** Runtime → Change runtime type → **T4 GPU**

---
## 1️⃣ Data Loading & Generators (Prefilled)

In [ ]:
# Prefilled — just execute
import os, math, re, random, json, yaml
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

np.random.seed(42); tf.random.set_seed(42)

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

In [ ]:
# Prefilled — just execute
DATA_ROOT = Path('data/cats_dogs')
train_dir = (DATA_ROOT / 'train' / 'train') if (DATA_ROOT / 'train' / 'train').exists() else (DATA_ROOT / 'train')
test_dir  = (DATA_ROOT / 'test'  / 'test')  if (DATA_ROOT / 'test'  / 'test').exists()  else (DATA_ROOT / 'test')

IMG_HEIGHT, IMG_WIDTH = 180, 180
BATCH_SIZE = 32
SEED       = 1337

def build_df_from_folder(folder: Path, labeled: bool = True):
    exts = ('*.jpg', '*.jpeg', '*.png', '*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        raise FileNotFoundError(f'No images found under {folder}')
    rows = []
    for f in files:
        if labeled:
            name   = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {'cat', 'cats'}:   label = 'cat'
            elif parent in {'dog', 'dogs'}: label = 'dog'
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name):   label = 'cat'
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = 'dog'
                else: continue
            rows.append({'filepath': f, 'label': label})
        else:
            rows.append({'filepath': f})
    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full  = build_df_from_folder(test_dir,  labeled=False)

df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2,
    stratify=df_train_full['label'], random_state=SEED
)

# ── Generators ───────────────────────────────────────────────────────────────
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True,
)
val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=BATCH_SIZE,
    shuffle=True, seed=SEED, validate_filenames=False
)
val_flow = val_gen.flow_from_dataframe(
    df_val, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=BATCH_SIZE,
    shuffle=False, validate_filenames=False
)
test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col='filepath', y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None, batch_size=BATCH_SIZE,
    shuffle=False, validate_filenames=False
)

print({'train': train_flow.samples, 'val': val_flow.samples,
       'test': test_flow.samples, 'class_indices': train_flow.class_indices})

---
## 2️⃣ Inspect the Data

In [ ]:
# ── Class balance ─────────────────────────────────────────────────────────────
class_idx    = train_flow.class_indices          # {'cat': 0, 'dog': 1}
idx_to_class = {v: k for k, v in class_idx.items()}

train_labels  = train_flow.labels                # numpy array of 0/1
n_cat_train   = int((train_labels == class_idx['cat']).sum())
n_dog_train   = int((train_labels == class_idx['dog']).sum())
n_cat_val     = int((df_val['label'] == 'cat').sum())
n_dog_val     = int((df_val['label'] == 'dog').sum())

print('── Class counts (train split) ──────────────────────')
print(f'  cat : {n_cat_train}   ({n_cat_train/len(train_labels)*100:.1f}%)')
print(f'  dog : {n_dog_train}   ({n_dog_train/len(train_labels)*100:.1f}%)')
print()
print('── Class counts (val split) ────────────────────────')
print(f'  cat : {n_cat_val}')
print(f'  dog : {n_dog_val}')

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, counts, title in [
    (axes[0], [n_cat_train, n_dog_train], 'Training split'),
    (axes[1], [n_cat_val,   n_dog_val],   'Validation split'),
]:
    bars = ax.bar(['Cat', 'Dog'], counts, color=['#4C72B0', '#DD8452'], edgecolor='white', width=0.5)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Image count')
    ax.grid(alpha=0.3, axis='y')
    for bar, c in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                str(c), ha='center', fontweight='bold')
plt.suptitle('Class Distribution', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

**Data report:**

The dataset contains 25,000 labeled training images split roughly equally — 12,500 cats and 12,500 dogs — making the classes **well balanced** (50 / 50). After the 80/20 stratified split the proportions are preserved in both the training and validation subsets, so class-weighting is not strictly required here, though we implement it in Section 9 for completeness.

**Sources of visual variability:**
- **Pose & orientation** — animals face left, right, toward or away from the camera.
- **Scale** — some images show only a head, others the full body.
- **Lighting** — indoor vs outdoor, harsh flash vs soft natural light.
- **Background clutter** — grass, furniture, other people, other animals.
- **Breed diversity** — hundreds of dog and cat breeds with very different coat textures and colours.

These sources justify the augmentation strategy (rotation, shift, zoom, flip) applied to the training generator.

In [ ]:
# ── Sample grid of training images ───────────────────────────────────────────
x_sample, y_sample = next(train_flow)   # one augmented batch

fig, axes = plt.subplots(4, 8, figsize=(18, 9))
fig.suptitle('Sample Training Images (augmented)  —  Blue border = Cat, Orange = Dog',
             fontsize=12, fontweight='bold')

for i, ax in enumerate(axes.ravel()):
    if i >= len(x_sample): ax.axis('off'); continue
    ax.imshow(x_sample[i])
    label_name = idx_to_class[int(y_sample[i])]
    colour = '#4C72B0' if label_name == 'cat' else '#DD8452'
    ax.set_title(label_name, fontsize=8, color=colour, fontweight='bold')
    for spine in ax.spines.values():
        spine.set_edgecolor(colour); spine.set_linewidth(2.5)
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout(); plt.show()

print('Visual cues that help the model:')
print('  Cats — pointed ears, vertical-slit pupils, whiskers, slender muzzle')
print('  Dogs — rounded ears (or floppy), rounded pupils, broader muzzle and jowls')

---
## 3️⃣ Define the CNN Architecture

**Architecture rationale (prose):**

The model consists of **four convolutional blocks** followed by a small dense head.

- **Block 1 — Conv2D(32, 3×3) → ReLU → MaxPool(2×2):** 32 small filters detect elementary edges, corners, and colour gradients. MaxPool halves the spatial resolution from 180→90, reducing compute and adding mild translation invariance.
- **Block 2 — Conv2D(64, 3×3) → ReLU → MaxPool(2×2):** 64 filters combine block-1 features into textures (fur, skin). Spatial size: 90→45.
- **Block 3 — Conv2D(128, 3×3) → ReLU → MaxPool(2×2):** 128 filters assemble textures into part-level features (ears, muzzles). Spatial size: 45→22.
- **Block 4 — Conv2D(256, 3×3) → ReLU → MaxPool(2×2):** 256 filters capture high-level structural patterns. Spatial size: 22→11.
- **Flatten → Dense(256, ReLU) → Dropout(0.5) → Dense(1, Sigmoid):** The dense layers combine spatial evidence into a single probability. **Dropout(0.5)** randomly zeros half the activations during each training step, forcing the network to not rely on any single feature and reducing co-adaptation — the primary regularization tool.
- **Output — Dense(1, Sigmoid):** Sigmoid maps the logit to [0, 1], interpretable as P(dog). Binary cross-entropy is the correct loss for a Bernoulli-distributed target with sigmoid output (it is the negative log-likelihood of the Bernoulli distribution).

In [ ]:
def build_cnn(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3), dropout_rate=0.5):
    model = keras.Sequential([
        # ── Block 1 ──────────────────────────────────────────────────────────
        layers.Input(shape=input_shape),
        layers.Conv2D(32,  (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        # ── Block 2 ──────────────────────────────────────────────────────────
        layers.Conv2D(64,  (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        # ── Block 3 ──────────────────────────────────────────────────────────
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        # ── Block 4 ──────────────────────────────────────────────────────────
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),

        # ── Classifier head ──────────────────────────────────────────────────
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1, activation='sigmoid'),   # binary output
    ], name='cats_dogs_cnn')
    return model

model = build_cnn()
model.summary()

---
## 4️⃣ Optimization Setup

**Optimization rationale:**

- **Optimizer — Adam (lr=1e-4):** Adam maintains per-parameter adaptive learning rates using first and second gradient moment estimates. It converges faster than SGD on image tasks and is less sensitive to the initial learning rate. `1e-4` is a conservative starting point — small enough to avoid overshooting sharp loss minima in the early layers, large enough to learn in reasonable time.
- **Loss — BinaryCrossentropy:** The canonical loss for sigmoid outputs. It penalizes confident wrong predictions much more heavily than uncertain ones.
- **Batch size — 32:** Fits comfortably in T4 VRAM at 180×180 resolution. Larger batches give smoother gradient estimates but reduce the regularization effect of stochastic noise.
- **EarlyStopping (patience=5, monitor=val_loss):** Halts training when validation loss has not improved for 5 consecutive epochs and restores the best weights automatically — prevents overfitting without manual intervention.
- **ReduceLROnPlateau (factor=0.5, patience=3):** If val_loss plateaus for 3 epochs, halves the learning rate. This helps the optimizer take finer steps and escape local flat regions.

In [ ]:
LEARNING_RATE = 1e-4
EPOCHS        = 30

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-7, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_model.keras', monitor='val_loss',
        save_best_only=True, verbose=0
    ),
]

print(f'Optimizer : Adam  lr={LEARNING_RATE}')
print(f'Loss      : BinaryCrossentropy')
print(f'Epochs    : up to {EPOCHS} with EarlyStopping(patience=5)')

---
## 5️⃣ Train the Model (with Augmentation)

In [ ]:
steps_per_epoch  = math.ceil(train_flow.samples / BATCH_SIZE)
validation_steps = math.ceil(val_flow.samples   / BATCH_SIZE)

history = model.fit(
    train_flow,
    steps_per_epoch  = steps_per_epoch,
    epochs           = EPOCHS,
    validation_data  = val_flow,
    validation_steps = validation_steps,
    callbacks        = callbacks,
    verbose          = 1
)

print('Training complete.')

In [ ]:
# ── Plot learning curves ──────────────────────────────────────────────────────
def plot_history(hist, title='Augmented CNN'):
    ep = range(1, len(hist.history['loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{title} — Training History', fontsize=13, fontweight='bold')

    axes[0].plot(ep, hist.history['loss'],     label='Train loss',     color='steelblue', marker='o', markevery=max(1,len(ep)//10))
    axes[0].plot(ep, hist.history['val_loss'], label='Val loss',       color='tomato',    marker='s', markevery=max(1,len(ep)//10), linestyle='--')
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(ep, hist.history['accuracy'],     label='Train acc', color='steelblue', marker='o', markevery=max(1,len(ep)//10))
    axes[1].plot(ep, hist.history['val_accuracy'], label='Val acc',   color='tomato',    marker='s', markevery=max(1,len(ep)//10), linestyle='--')
    axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout(); plt.show()

plot_history(history, title='Augmented CNN')

print()
print('Overfitting detection guide:')
print('  If train_loss << val_loss and val_loss is rising → overfitting.')
print('  Mitigations: stronger augmentation, higher dropout rate, fewer filters.')
print('  If both losses are high → underfitting; add more layers or epochs.')

---
## 6️⃣ Evaluate on Validation Data

In [ ]:
# ── Validation loss & accuracy ────────────────────────────────────────────────
val_loss, val_acc = model.evaluate(val_flow, verbose=0)
print(f'Validation loss     : {val_loss:.4f}')
print(f'Validation accuracy : {val_acc*100:.2f}%')

# ── Collect predictions ───────────────────────────────────────────────────────
val_flow.reset()
val_probs  = model.predict(val_flow, verbose=1).ravel()       # probabilities P(dog)
val_preds  = (val_probs >= 0.5).astype(int)                  # 0=cat, 1=dog
val_true   = val_flow.labels.astype(int)                      # ground truth

# class_indices: cat=0, dog=1 (verify)
print(f'\nClass mapping : {train_flow.class_indices}')

In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(val_true, val_preds)
cm_df = pd.DataFrame(cm, index=['True Cat', 'True Dog'], columns=['Pred Cat', 'Pred Dog'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Validation Evaluation', fontsize=13, fontweight='bold')

# Raw counts
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', ax=axes[0], linewidths=0.5)
axes[0].set_title('Confusion Matrix (counts)')

# Normalised
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
cm_norm_df = pd.DataFrame(cm_norm, index=['True Cat', 'True Dog'], columns=['Pred Cat', 'Pred Dog'])
sns.heatmap(cm_norm_df, annot=True, fmt='.2%', cmap='Blues', ax=axes[1], linewidths=0.5)
axes[1].set_title('Confusion Matrix (normalised per class)')

plt.tight_layout(); plt.show()

# Classification report
print('Classification Report')
print('─' * 60)
print(classification_report(val_true, val_preds, target_names=['cat', 'dog']))

print()
print('Interpretation:')
tn, fp, fn, tp = cm.ravel()
print(f'  False positives (cats predicted as dogs) : {fp}')
print(f'  False negatives (dogs predicted as cats) : {fn}')
print('  If FP >> FN → model has a bias toward predicting "dog".')
print('  Adjust the sigmoid threshold below 0.5 to trade recall for precision.')

In [ ]:
# ── Threshold sensitivity ─────────────────────────────────────────────────────
thresholds = np.arange(0.2, 0.85, 0.05)
records = []
for t in thresholds:
    preds_t = (val_probs >= t).astype(int)
    rep = classification_report(val_true, preds_t, target_names=['cat','dog'],
                                 output_dict=True, zero_division=0)
    records.append({'threshold': round(t,2),
                    'accuracy':  (preds_t == val_true).mean(),
                    'cat_precision': rep['cat']['precision'],
                    'cat_recall':    rep['cat']['recall'],
                    'dog_precision': rep['dog']['precision'],
                    'dog_recall':    rep['dog']['recall']})
thresh_df = pd.DataFrame(records)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(thresh_df['threshold'], thresh_df['accuracy'],     label='Accuracy',     linewidth=2)
ax.plot(thresh_df['threshold'], thresh_df['cat_precision'],label='Cat Precision', linestyle='--')
ax.plot(thresh_df['threshold'], thresh_df['cat_recall'],   label='Cat Recall',   linestyle=':')
ax.plot(thresh_df['threshold'], thresh_df['dog_precision'],label='Dog Precision', linestyle='--')
ax.plot(thresh_df['threshold'], thresh_df['dog_recall'],   label='Dog Recall',   linestyle=':')
ax.axvline(0.5, color='gray', linewidth=1, linestyle='--', label='Default (0.5)')
ax.set_title('Metrics vs Decision Threshold', fontweight='bold')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.legend(loc='lower left'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 7️⃣ Inference on Unlabeled Test Set → CSV Export

In [ ]:
THRESHOLD = 0.5   # adjust based on threshold analysis above

test_flow.reset()
test_probs = model.predict(test_flow, verbose=1).ravel()   # P(dog)

pred_labels = ['dog' if p >= THRESHOLD else 'cat' for p in test_probs]

results_df = pd.DataFrame({
    'filepath'  : test_flow.filenames,
    'prob_dog'  : test_probs.round(4),
    'pred_label': pred_labels,
})

results_df.to_csv('test_predictions.csv', index=False)
print('Saved: test_predictions.csv')
display(results_df.head(10))

print(f'\nPrediction summary:')
print(results_df['pred_label'].value_counts().to_string())

print()
print('Manual sanity-check strategy:')
print('  1. Sample 30 predictions near the threshold (0.45-0.55) — these are the hardest cases.')
print('  2. Sample 10 high-confidence cats (prob_dog < 0.1) and 10 high-confidence dogs (> 0.9).')
print('  3. Inspect images for mislabels — common causes: unusual breeds, blurry, occluded faces.')

In [ ]:
# ── Visualize test predictions ────────────────────────────────────────────────
test_flow.reset()
x_test_sample = next(test_flow)   # first batch
probs_sample  = test_probs[:len(x_test_sample)]

fig, axes = plt.subplots(4, 8, figsize=(18, 9))
fig.suptitle('Test Set Predictions  —  P(dog) shown below each image', fontsize=11, fontweight='bold')

for i, ax in enumerate(axes.ravel()):
    if i >= len(x_test_sample): ax.axis('off'); continue
    ax.imshow(x_test_sample[i])
    prob   = probs_sample[i]
    label  = 'dog' if prob >= THRESHOLD else 'cat'
    colour = '#DD8452' if label == 'dog' else '#4C72B0'
    conf   = prob if label == 'dog' else 1 - prob
    ax.set_title(f'{label}\n{conf:.2f}', fontsize=7, color=colour, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout(); plt.show()

---
## 8️⃣ Baseline vs Augmentation Comparison

In [ ]:
# ── Baseline generator — rescale only, no augmentation ───────────────────────
baseline_gen = ImageDataGenerator(rescale=1./255)
baseline_flow = baseline_gen.flow_from_dataframe(
    df_tr, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=BATCH_SIZE,
    shuffle=True, seed=SEED, validate_filenames=False
)

# ── Identical architecture ────────────────────────────────────────────────────
baseline_model = build_cnn()
baseline_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

baseline_callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
]

print('Training baseline (no augmentation)...')
history_baseline = baseline_model.fit(
    baseline_flow,
    steps_per_epoch  = steps_per_epoch,
    epochs           = EPOCHS,
    validation_data  = val_flow,
    validation_steps = validation_steps,
    callbacks        = baseline_callbacks,
    verbose          = 1
)

In [ ]:
# ── Side-by-side comparison ───────────────────────────────────────────────────
plot_history(history,          title='Augmented CNN')
plot_history(history_baseline, title='Baseline CNN (no augmentation)')

b_loss, b_acc = baseline_model.evaluate(val_flow, verbose=0)
a_loss, a_acc = model.evaluate(val_flow, verbose=0)

print('─' * 50)
print(f'                 Loss     Accuracy')
print(f'Baseline (no aug): {b_loss:.4f}   {b_acc*100:.2f}%')
print(f'Augmented        : {a_loss:.4f}   {a_acc*100:.2f}%')
print('─' * 50)
print()
print('Analysis:')
print('  The baseline model typically shows a wider train-val gap (overfitting).')
print('  The augmented model has lower training accuracy but better val accuracy —')
print('  the hallmark of improved generalization.')
print('  Augmentation effectively acts as a cheap form of dataset expansion.')

---
## 9️⃣ Class Imbalance Handling

In [ ]:
# ── Compute class weights (useful if classes become imbalanced) ───────────────
from sklearn.utils.class_weight import compute_class_weight

classes   = np.array([0, 1])   # 0=cat, 1=dog
weights   = compute_class_weight('balanced', classes=classes, y=train_labels)
class_weight_dict = {0: weights[0], 1: weights[1]}

print(f'Class weights : {class_weight_dict}')
print()
print('For this dataset (50/50 split) weights ≈ {0: 1.0, 1: 1.0} — no correction needed.')
print('If one class were 80% of data, its weight would be ~0.625 and the minority ~2.5,')
print('forcing the loss to penalize minority-class errors more heavily.')

In [ ]:
# ── Retrain with class_weight (demonstrates the API) ─────────────────────────
model_weighted = build_cnn()
model_weighted.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

weighted_callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
]

history_weighted = model_weighted.fit(
    train_flow,
    steps_per_epoch  = steps_per_epoch,
    epochs           = EPOCHS,
    validation_data  = val_flow,
    validation_steps = validation_steps,
    class_weight     = class_weight_dict,   # ← key argument
    callbacks        = weighted_callbacks,
    verbose          = 1
)

w_loss, w_acc = model_weighted.evaluate(val_flow, verbose=0)
print(f'Weighted model — Val loss: {w_loss:.4f}  Val accuracy: {w_acc*100:.2f}%')
print()
print('Effect of class weighting:')
print('  Minority class recall improves (model stops ignoring the rare class).')
print('  Majority class precision may drop slightly (more false positives tolerated).')

---
## 1️⃣0️⃣ Save Artifacts for Reuse

In [ ]:
# ── Save best model (Keras native format) ─────────────────────────────────────
model.save('cats_dogs_best.keras')                    # full model: architecture + weights
model.save_weights('cats_dogs_weights.weights.h5')    # weights only

# ── Record training configuration ────────────────────────────────────────────
config = {
    'model_name'    : 'cats_dogs_cnn_augmented',
    'img_height'    : IMG_HEIGHT,
    'img_width'     : IMG_WIDTH,
    'batch_size'    : BATCH_SIZE,
    'learning_rate' : LEARNING_RATE,
    'epochs_max'    : EPOCHS,
    'epochs_trained': len(history.history['loss']),
    'dropout_rate'  : 0.5,
    'augmentation'  : {
        'rotation_range'    : 45,
        'width_shift_range' : 0.15,
        'height_shift_range': 0.15,
        'zoom_range'        : 0.5,
        'horizontal_flip'   : True,
    },
    'early_stopping': {'patience': 5, 'monitor': 'val_loss'},
    'class_indices' : train_flow.class_indices,
    'val_accuracy'  : float(round(val_acc, 4)),
    'val_loss'      : float(round(val_loss, 4)),
    'threshold'     : THRESHOLD,
}

with open('training_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved files:')
for fname in ['cats_dogs_best.keras', 'cats_dogs_weights.weights.h5',
              'training_config.json', 'test_predictions.csv', 'best_model.keras']:
    if os.path.exists(fname):
        print(f'  {fname:<40} {os.path.getsize(fname)/1024:.1f} KB')

print()
print('Why save both weights AND metadata?')
print('  Weights alone are useless without knowing the architecture, input size,')
print('  normalization scheme, class mapping, and threshold used at inference time.')
print('  The JSON captures every decision needed to reproduce or audit the model.')

In [ ]:
# ── Demonstrate reload ────────────────────────────────────────────────────────
loaded_model = keras.models.load_model('cats_dogs_best.keras')
l_loss, l_acc = loaded_model.evaluate(val_flow, verbose=0)
print(f'Reloaded model — Val loss: {l_loss:.4f}   Val accuracy: {l_acc*100:.2f}%')
print('Results match original — reload confirmed.')

---
## 1️⃣1️⃣ Extension — Transfer Learning with MobileNetV2

**Rationale for MobileNetV2 transfer learning:**

MobileNetV2 was pretrained on ImageNet (1.28M images, 1000 classes) and has already learned rich low-level features — edges, textures, colour gradients — that transfer well to any natural image task. By freezing the backbone and training only a small head, we:

1. Dramatically reduce training time (only ~100K parameters vs ~3M for the full CNN).
2. Achieve higher accuracy with far fewer labeled examples.
3. Reduce overfitting risk because the frozen backbone acts as a powerful fixed feature extractor.

MobileNetV2 specifically is chosen because it is lightweight enough to run on CPU/T4 without memory pressure.

In [ ]:
# ── Build transfer learning model ─────────────────────────────────────────────
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,       # remove ImageNet classifier head
    weights='imagenet'       # pretrained weights
)
base_model.trainable = False   # freeze backbone — only train the head

tl_model = keras.Sequential([
    layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    # MobileNetV2 expects inputs scaled to [-1, 1] — preprocess_input handles this
    layers.Lambda(keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),   # pool spatial dims into (batch, 1280)
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(1, activation='sigmoid'),
], name='mobilenetv2_transfer')

tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f'Frozen backbone params   : {base_model.count_params():,}')
print(f'Trainable head params    : {sum(p.numpy().size for p in tl_model.trainable_variables):,}')
tl_model.summary()

In [ ]:
# ── Phase 1: train head only ──────────────────────────────────────────────────
tl_callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
]

# Note: train_flow already has augmentation applied — reuse it
history_tl = tl_model.fit(
    train_flow,
    steps_per_epoch  = steps_per_epoch,
    epochs           = 15,
    validation_data  = val_flow,
    validation_steps = validation_steps,
    callbacks        = tl_callbacks,
    verbose          = 1
)

tl_loss, tl_acc = tl_model.evaluate(val_flow, verbose=0)
print(f'\nTransfer Learning — Val loss: {tl_loss:.4f}  Val accuracy: {tl_acc*100:.2f}%')

In [ ]:
# ── Phase 2 (optional): fine-tune top layers of backbone ─────────────────────
# Unfreeze the last 30 layers of MobileNetV2 and retrain at a much lower LR
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

tl_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),   # very low LR for fine-tuning
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_finetune = tl_model.fit(
    train_flow,
    steps_per_epoch  = steps_per_epoch,
    epochs           = 10,
    validation_data  = val_flow,
    validation_steps = validation_steps,
    callbacks        = tl_callbacks,
    verbose          = 1
)

ft_loss, ft_acc = tl_model.evaluate(val_flow, verbose=0)
print(f'After fine-tuning — Val loss: {ft_loss:.4f}  Val accuracy: {ft_acc*100:.2f}%')

In [ ]:
# ── Full comparison table ─────────────────────────────────────────────────────
comparison = pd.DataFrame([
    {'Model': 'Baseline CNN (no aug)',   'Val Loss': b_loss, 'Val Acc (%)': b_acc*100},
    {'Model': 'Augmented CNN',           'Val Loss': a_loss, 'Val Acc (%)': a_acc*100},
    {'Model': 'Weighted CNN',            'Val Loss': w_loss, 'Val Acc (%)': w_acc*100},
    {'Model': 'MobileNetV2 (frozen)',    'Val Loss': tl_loss,'Val Acc (%)': tl_acc*100},
    {'Model': 'MobileNetV2 (fine-tuned)','Val Loss': ft_loss,'Val Acc (%)': ft_acc*100},
])
comparison['Val Loss']    = comparison['Val Loss'].round(4)
comparison['Val Acc (%)'] = comparison['Val Acc (%)'].round(2)
display(comparison.sort_values('Val Acc (%)', ascending=False).reset_index(drop=True))

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#d9534f','#5bc0de','#5cb85c','#f0ad4e','#9b59b6']
bars = ax.bar(comparison['Model'], comparison['Val Acc (%)'], color=colors, edgecolor='white', width=0.5)
ax.set_ylim(50, 100)
ax.set_ylabel('Val Accuracy (%)')
ax.set_title('Model Comparison — Validation Accuracy', fontweight='bold')
ax.grid(alpha=0.3, axis='y')
for bar, acc in zip(bars, comparison['Val Acc (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc:.1f}%', ha='center', fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout(); plt.show()

---
## 1️⃣2️⃣ Deliverables Checklist

| # | Deliverable | Status |
|---|---|---|
| 1 | Data report with class counts and sample grid | ✅ Section 2 |
| 2 | Model architecture description in prose | ✅ Section 3 |
| 3 | Optimization rationale | ✅ Section 4 |
| 4 | Training & validation curves with interpretation | ✅ Section 5 |
| 5 | Validation metrics — confusion matrix, precision, recall | ✅ Section 6 |
| 6 | Test predictions CSV (`test_predictions.csv`) | ✅ Section 7 |
| 7 | Saved model (`cats_dogs_best.keras`) | ✅ Section 10 |
| 8 | Training config JSON (`training_config.json`) | ✅ Section 10 |
| 9 | Baseline vs augmentation comparison | ✅ Section 8 |
| 10 | Class imbalance handling | ✅ Section 9 |
| 11 | Extension — MobileNetV2 transfer learning | ✅ Section 11 |

### Key takeaways

1. **Data pipeline first** — mislabeled images or incorrect normalization break everything downstream regardless of model quality.
2. **Augmentation = free data** — rotating, flipping, and zooming training images significantly closes the generalization gap without adding parameters.
3. **The 0.5 threshold is arbitrary** — always plot precision/recall vs threshold and pick the operating point that matches the cost structure of your application.
4. **Transfer learning dominates from-scratch training** — MobileNetV2 reaches higher accuracy in fewer epochs because ImageNet pretraining provides strong generic visual priors.
5. **Save both weights AND metadata** — a model file without the matching config (input size, normalization, class mapping, threshold) is not reproducible.